LAB 7

In [20]:
con.execute("CREATE OR REPLACE TABLE silver_transactions AS SELECT * FROM read_parquet('bigdata/silver/transactions_enriched.parquet')")

con.execute("""
    CREATE OR REPLACE TABLE gold_fraud_risk AS
    SELECT
        segment,
        COUNT(*) AS total_transacoes,
        SUM(amount) AS valor_total,
        ROUND(AVG(amount), 2) AS ticket_medio,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS qtd_fraudes,
        ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 2) AS taxa_fraude_pct,
        SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS valor_em_risco
    FROM silver_transactions GROUP BY segment
""")
print(con.execute("SELECT * FROM gold_fraud_risk ORDER BY taxa_fraude_pct DESC").fetchdf())

     segment  total_transacoes   valor_total  ticket_medio  qtd_fraudes  \
0  High-Risk              9155  1.664640e+06        181.83        705.0   
1   Standard             29689  5.479400e+06        184.56        655.0   
2    Premium             61156  1.123504e+07        183.71        473.0   

   taxa_fraude_pct  valor_em_risco  
0             7.70   126451.069108  
1             2.21   106828.419109  
2             0.77    81308.437996  


In [21]:
con.execute("""
    CREATE OR REPLACE TABLE gold_daily_metrics AS
    SELECT year, month, day, COUNT(*) AS transacoes,
           SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraudes
    FROM silver_transactions GROUP BY year, month, day
""")
print(con.execute("SELECT * FROM gold_daily_metrics ORDER BY fraudes DESC LIMIT 5").fetchdf())

   year  month  day  transacoes  fraudes
0  2023      4   27         353     16.0
1  2023      5   21         359     13.0
2  2023     12   22         335     13.0
3  2024      5   23         350     13.0
4  2024      5   25         310     12.0


In [22]:
con.execute("COPY gold_fraud_risk TO 'bigdata/gold/fraud_risk.parquet' (FORMAT PARQUET)")
con.execute("COPY gold_daily_metrics TO 'bigdata/gold/daily_metrics.parquet' (FORMAT PARQUET)")
con.execute("COPY gold_fraud_risk TO 'bigdata/gold/fraud_risk.csv' (HEADER, DELIMITER ',')")
print("Gold salva em bigdata/gold/")

Gold salva em bigdata/gold/
